# CapCap GPU Server - All in One (Bóc Băng, Tách Giọng, TTS)
Sổ tay này cài đặt **toàn bộ hệ sinh thái của CapCap** trên Google Colab. Bằng cách chạy file này, bạn chỉ cần lấy 1 URL duy nhất để sử dụng cho **tất cả** tính năng nặng (Whisper, Demucs UVR, Đọc TTS, v.v.).

**Lưu ý:** Việc cài đặt toàn bộ thư viện sẽ mất khoảng **7-10 phút** cho lần chạy đầu tiên. Hãy kiên nhẫn!

**Hướng dẫn:**
1. Vào menu **Runtime** > **Change runtime type** > Chọn **T4 GPU**.
2. Bấm nút **Play** ở ô bên dưới để chạy mã.
3. Đợi quá trình cài đặt hoàn tất. Khi thành công, hệ thống sẽ in ra URL (đuôi `.trycloudflare.com`) và Token.
4. Mở giao diện **Settings** của CapCap trên máy tính của bạn, tích chọn tính năng Colab, nhập URL và Token vào rồi dùng.

In [ ]:
import os
import subprocess
import time
import secrets
import IPython.display as display

print("1. Tải mã nguồn KOVA-STUDIO từ GitHub...")
!git clone https://github.com/khoinguyen59/KOVA-STUDIO.git /content/KOVA-STUDIO
%cd /content/KOVA-STUDIO

print("\n2. Cài đặt các thư viện lõi (Sẽ mất khoảng 5-10 phút, vui lòng kiên nhẫn)...")
!pip install -r requirements-local.txt > /dev/null 2>&1
# Colab is Linux; this is used by the remote Edge-TTS MP3-to-WAV conversion.
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1
assert subprocess.run(["ffmpeg", "-version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0, "FFmpeg is required for TTS"

print("\n3. Cài đặt Cloudflare Tunnel...")
!curl -s -L --output cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared.deb > /dev/null 2>&1

print("\n4. Thiết lập Token bảo mật...")
TOKEN = secrets.token_urlsafe(32)
env_vars = os.environ.copy()
env_vars["CAPCAP_REMOTE_API_TOKEN"] = TOKEN
# The server executes models in this Colab runtime.  Only the Windows client is remote.
env_vars["CAPCAP_RUNTIME_PROFILE"] = "local"
env_vars["CAPCAP_DEVICE"] = "cuda"
env_vars["CAPCAP_REQUIRE_GPU"] = "1"
env_vars.pop("CAPCAP_REMOTE_API_URL", None)
env_vars["PYTHONPATH"] = "/content/KOVA-STUDIO:/content/KOVA-STUDIO/app"

print("\n5. Khởi động CapCap All-in-One Server...")
# Chạy server gốc của CapCap
server_process = subprocess.Popen(
    ["python", "app/remote_api_server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env_vars
)
time.sleep(8)

print("\n6. Thiết lập kết nối Cloudflare...")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://0.0.0.0:8765", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = ""
while True:
    line = tunnel.stdout.readline()
    if not line:
        break
    if "https://" in line and ".trycloudflare.com" in line:
        import re
        match = re.search(r"https://[^\s\"']+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break

display.clear_output()
print("\n" + "="*70)
print("✅ MÁY CHỦ COLAB ALL-IN-ONE ĐÃ SẴN SÀNG ✅")
print("Hãy copy 2 dòng sau và dán vào phần cài đặt Colab trên máy tính:")
print("="*70)
print(f"URL:   {public_url}")
print(f"Token: {TOKEN}")
print("="*70)
print("\nBạn có thể để tab này chạy nền. Đừng đóng trình duyệt nhé!\n")

# Hiển thị log của server liên tục
try:
    for line in iter(server_process.stdout.readline, ""):
        print(line, end="")
except KeyboardInterrupt:
    print("\nĐã dừng server.")
